# Task 2: Model Comparison, Final Judgement, and Prediction

This notebook collects the exported results from the baseline and three candidate models. It does not retrain any model.


## 1. Setup

Run Notebooks 1–4 first. This notebook requires their saved validation predictions and trained model files.


In [ ]:
%matplotlib inline
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import confusion_matrix
from torchvision.models import densenet121, efficientnet_b0

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "pyproject.toml").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the repository root")
sys.path.insert(0, str(REPO_ROOT))

from src.preprocessing import TEST_IMAGE_DIR, IMAGE_TARGET_SIZE, load_image_array
from src.task2_utils import (
    TASK2_OUTPUT_DIR, TASK2_PREDICTION_PATH, TASK2_PREPROCESSED_DIR,
    evaluate_predictions, extract_visual_features, load_validation_predictions,
    per_class_table,
)
sns.set_theme(style="whitegrid", context="notebook")


## 2. Compare All Models

Macro-F1 is the primary selection metric because every season contributes equally despite class imbalance. Accuracy, balanced accuracy, weighted F1 and top-2 accuracy provide supporting evidence. The baselines are references only and cannot be selected as the final trained model.


In [ ]:
setup = json.loads((TASK2_OUTPUT_DIR / "setup" / "config.json").read_text())
CLASSES = setup["classes"]
TARGET = setup["target"]

frames = []
for folder, label in [
    ("setup", "Baseline: majority class"),
    ("random_forest", "Random Forest"),
    ("efficientnet_b0", "EfficientNet-B0"),
    ("densenet121", "DenseNet-121"),
]:
    path = TASK2_OUTPUT_DIR / folder / "validation_predictions.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run that model notebook first.")
    frames.append((label, folder, load_validation_predictions(path, CLASSES)))

reference_ids = frames[0][2]["id"].to_numpy()
reference_truth = frames[0][2]["true_index"].to_numpy()
for label, _, frame in frames[1:]:
    if not np.array_equal(frame["id"].to_numpy(), reference_ids):
        raise ValueError(f"{label} validation rows do not match Notebook 1. Rerun that model notebook.")
    if not np.array_equal(frame["true_index"].to_numpy(), reference_truth):
        raise ValueError(f"{label} validation labels do not match Notebook 1. Rerun that model notebook.")

rows = []
for label, folder, frame in frames:
    scores = frame[[f"score_{name}" for name in CLASSES]].to_numpy()
    rows.append(evaluate_predictions(
        frame["true_index"].to_numpy(), frame["predicted_index"].to_numpy(), scores, label,
    ))
model_comparison = pd.DataFrame(rows).sort_values("Macro-F1", ascending=False)
display(model_comparison.style.format({
    column: "{:.4f}" for column in model_comparison.columns if column != "Model"
}).background_gradient(subset=["Macro-F1"], cmap="Blues"))

candidate_names = {"Random Forest", "EfficientNet-B0", "DenseNet-121"}
candidate_results = model_comparison[model_comparison["Model"].isin(candidate_names)]
FINAL_NAME = candidate_results.iloc[0]["Model"]
winner = next(item for item in frames if item[0] == FINAL_NAME)
final_validation = winner[2]
print("Metric-table leader:", FINAL_NAME)


## 3. Ultimate Judgement

The winner is judged from validation performance first, then checked for class-level weaknesses, confidence reliability and deployment cost. Architecture reputation or ImageNet accuracy is not used to choose the model.

### 3.1 Per-season performance and confusion

This shows whether the selected model performs consistently across Spring, Summer, Fall and Winter rather than succeeding mainly on the largest class.


In [ ]:
y_true = final_validation["true_index"].to_numpy()
y_pred = final_validation["predicted_index"].to_numpy()
score_columns = [f"score_{name}" for name in CLASSES]
final_scores = final_validation[score_columns].to_numpy()

season_results = per_class_table(y_true, y_pred, CLASSES)
display(season_results.style.format({"Precision": "{:.3f}", "Recall": "{:.3f}", "F1": "{:.3f}"}))

matrix = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASSES)), normalize="true")
plt.figure(figsize=(7, 6))
sns.heatmap(matrix, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, vmin=0, vmax=1)
plt.title(f"{FINAL_NAME}: row-normalised confusion matrix")
plt.xlabel("Predicted season")
plt.ylabel("True season")
plt.tight_layout()
plt.show()


### 3.2 Calibration

Calibration checks whether the selected model's confidence matches its observed accuracy. A lower Expected Calibration Error (ECE) is better.


In [ ]:
def expected_calibration_error(probabilities, truth, n_bins=10):
    confidence = probabilities.max(axis=1)
    predicted = probabilities.argmax(axis=1)
    edges = np.linspace(0, 1, n_bins + 1)
    rows = []
    ece = 0.0
    for lower, upper in zip(edges[:-1], edges[1:]):
        selected = (confidence > lower) & (confidence <= upper)
        if not selected.any():
            continue
        bin_accuracy = (predicted[selected] == truth[selected]).mean()
        bin_confidence = confidence[selected].mean()
        ece += selected.mean() * abs(bin_accuracy - bin_confidence)
        rows.append({"confidence": bin_confidence, "accuracy": bin_accuracy,
                     "count": int(selected.sum())})
    return ece, pd.DataFrame(rows)

ece, calibration = expected_calibration_error(final_scores, y_true)
print(f"Expected Calibration Error: {ece:.4f}")
plt.figure(figsize=(5.5, 5))
plt.plot([0, 1], [0, 1], "--", color="#6b7280", label="Perfect calibration")
plt.plot(calibration["confidence"], calibration["accuracy"], marker="o", label=FINAL_NAME)
plt.xlabel("Mean confidence")
plt.ylabel("Observed accuracy")
plt.title("Validation calibration")
plt.legend()
plt.tight_layout()
plt.show()


### 3.3 Deployment cost and model attention

Model size is a secondary consideration. It should break a close performance tie, not replace the validation metrics. For a selected neural model, optional saliency maps provide a simple check that predictions respond to the product rather than only the background.


In [ ]:
model_paths = {
    "Random Forest": REPO_ROOT / "models" / "task2_random_forest.joblib",
    "EfficientNet-B0": REPO_ROOT / "models" / "task2_efficientnet_b0.pt",
    "DenseNet-121": REPO_ROOT / "models" / "task2_densenet121.pt",
}
efficientnet_for_count = efficientnet_b0(weights=None)
efficientnet_for_count.classifier[1] = nn.Linear(
    efficientnet_for_count.classifier[1].in_features, len(CLASSES)
)
densenet_for_count = densenet121(weights=None)
densenet_for_count.classifier = nn.Linear(
    densenet_for_count.classifier.in_features, len(CLASSES)
)
parameter_counts = {
    "Random Forest": np.nan,
    "EfficientNet-B0": sum(p.numel() for p in efficientnet_for_count.parameters()),
    "DenseNet-121": sum(p.numel() for p in densenet_for_count.parameters()),
}
deployment = pd.DataFrame([
    {"Model": name, "Parameters": parameter_counts[name],
     "Saved size (MB)": path.stat().st_size / 1e6 if path.exists() else np.nan}
    for name, path in model_paths.items()
])
display(deployment.style.format({"Parameters": "{:,.0f}", "Saved size (MB)": "{:.1f}"}))

RUN_SALIENCY = True

def load_selected_neural(name):
    if name == "EfficientNet-B0":
        model = efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASSES))
    elif name == "DenseNet-121":
        model = densenet121(weights=None)
        model.classifier = nn.Linear(model.classifier.in_features, len(CLASSES))
    else:
        raise ValueError(f"{name} is not a neural candidate")
    blob = torch.load(model_paths[name], map_location="cpu", weights_only=False)
    model.load_state_dict(blob["state_dict"])
    return model

selected_neural_model = None
if FINAL_NAME in {"EfficientNet-B0", "DenseNet-121"}:
    print("Selected neural-model parameters:", f"{parameter_counts[FINAL_NAME]:,.0f}")
    if RUN_SALIENCY:
        saliency_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        selected_neural_model = load_selected_neural(FINAL_NAME).to(saliency_device).eval()
        validation_images = np.load(
            TASK2_PREPROCESSED_DIR / "validation_images.npy", mmap_mode="r"
        )
        chosen = np.random.RandomState(setup["random_state"]).choice(
            len(y_true), min(6, len(y_true)), replace=False
        )
        raw = torch.from_numpy(np.asarray(validation_images[chosen]).copy()).to(saliency_device)
        raw = raw.permute(0, 3, 1, 2).float() / 255
        mean_for_saliency = torch.tensor(
            setup["normalisation_mean"], device=saliency_device
        ).view(1, 3, 1, 1)
        std_for_saliency = torch.tensor(
            setup["normalisation_std"], device=saliency_device
        ).view(1, 3, 1, 1)
        fig, axes = plt.subplots(2, len(chosen), figsize=(2.2 * len(chosen), 4.5))
        axes = np.asarray(axes).reshape(2, -1)
        for column, position in enumerate(chosen):
            image = ((raw[column:column + 1] - mean_for_saliency) /
                     std_for_saliency).clone().requires_grad_(True)
            logits = selected_neural_model(image)
            logits[0, logits.argmax(dim=1).item()].backward()
            saliency = image.grad.abs().max(dim=1).values.squeeze().detach().cpu()
            axes[0, column].imshow(raw[column].permute(1, 2, 0).cpu())
            axes[1, column].imshow(saliency, cmap="inferno")
            axes[0, column].set_title(
                f"true {CLASSES[y_true[position]]}\npred {CLASSES[y_pred[position]]}",
                fontsize=8,
            )
            axes[0, column].axis("off")
            axes[1, column].axis("off")
        plt.tight_layout()
        plt.show()
else:
    print("Random Forest uses permutation importance from Notebook 2 instead of saliency.")


### 3.4 The judgement

Select the candidate with the highest validation Macro-F1. Use the remaining metrics, per-season results, calibration and deployment cost to explain the decision and any trade-offs. A very small score margin should be described as a narrow result rather than proof that one architecture is universally better.


In [ ]:
winner_row = candidate_results.iloc[0]
runner_up = candidate_results.iloc[1]
margin = winner_row["Macro-F1"] - runner_up["Macro-F1"]
weakest = season_results.sort_values("F1").iloc[0]

judgement = pd.DataFrame([{
    "Selected model": FINAL_NAME,
    "Macro-F1": winner_row["Macro-F1"],
    "Margin over runner-up": margin,
    "Weakest season": weakest["Season"],
    "Weakest-season F1": weakest["F1"],
    "Calibration ECE": ece,
}])
display(judgement.style.format({
    "Macro-F1": "{:.4f}", "Margin over runner-up": "{:.4f}",
    "Weakest-season F1": "{:.4f}", "Calibration ECE": "{:.4f}",
}))


## 4. Persist the Comparison and Produce Final Predictions

The comparison and judgement are stored under `outputs/task2/analysis`. The required test predictions are written separately to `predictions/task2_season_predictions.csv`.


In [ ]:
analysis_dir = TASK2_OUTPUT_DIR / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)
model_comparison.to_csv(analysis_dir / "model_comparison.csv", index=False)
season_results.to_csv(analysis_dir / "per_season_results.csv", index=False)
calibration.to_csv(analysis_dir / "calibration.csv", index=False)
deployment.to_csv(analysis_dir / "deployment_cost.csv", index=False)
judgement.to_json(analysis_dir / "final_judgement.json", orient="records", indent=2)
print("Saved analysis outputs:", analysis_dir)


In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
template_path = REPO_ROOT / "datasets" / "test" / "styles_prediction.csv"
template = pd.read_csv(template_path)
test_paths = [Path(TEST_IMAGE_DIR) / f"{image_id}.jpg" for image_id in template["id"]]
missing = [path for path in test_paths if not path.exists()]
assert not missing, f"{len(missing)} test images are missing"

if FINAL_NAME == "Random Forest":
    model = joblib.load(model_paths[FINAL_NAME])
    features = np.vstack([
        extract_visual_features(load_image_array(path, IMAGE_TARGET_SIZE, False))
        for path in test_paths
    ])
    scores = model.predict_proba(features)
    predicted = model.classes_[scores.argmax(axis=1)]
else:
    model = (selected_neural_model.to(DEVICE) if selected_neural_model is not None
             else load_selected_neural(FINAL_NAME).to(DEVICE)).eval()
    mean = torch.tensor(setup["normalisation_mean"], device=DEVICE).view(1, 3, 1, 1)
    std = torch.tensor(setup["normalisation_std"], device=DEVICE).view(1, 3, 1, 1)
    chunks = []
    with torch.no_grad():
        for start in range(0, len(test_paths), 256):
            arrays = np.stack([load_image_array(path, IMAGE_TARGET_SIZE, False)
                               for path in test_paths[start:start + 256]])
            images = torch.from_numpy(arrays).to(DEVICE).permute(0, 3, 1, 2).float() / 255
            chunks.append(model((images - mean) / std).float().cpu())
    scores = torch.cat(chunks).numpy()
    predicted = scores.argmax(axis=1)

predictions = template.copy()
predictions[TARGET] = [CLASSES[index] for index in predicted]
assert predictions[TARGET].notna().all() and len(predictions) == len(template)
TASK2_PREDICTION_PATH.parent.mkdir(parents=True, exist_ok=True)
predictions.to_csv(TASK2_PREDICTION_PATH, index=False)
print("Written:", TASK2_PREDICTION_PATH)
display(predictions.head())


## 5. Decision Log, Limitations, and What to Tune Next

- The same leakage-safe split and metrics are used for every model.
- Metadata is not used as model input because the prediction file supplies only image IDs.
- Macro-F1 is the primary selection metric; deployment cost is secondary.
- Season can be weakly visible from product images, so errors may reflect ambiguous labels rather than only model capacity.
- If further tuning is allowed, tune the winning model using the fixed validation protocol and document every changed setting.
